# 02 Ingest — Azure Storage (parquet · full & incremental)

**Doel:** Lees alle actieve `azurestorage`-rijen uit de control table en laad de
bijbehorende parquet-bestanden als Delta-tabellen in `STAGING_AZURESTORAGE`.

**Laadstrategieën (gestuurd door `load_type` in de control table):**

| `load_type` | Gedrag |
|---|---|
| `full` | Alle bestanden opnieuw inlezen en de doeltabel volledig overschrijven |
| `incremental` | Auto Loader (`cloudFiles`) — alleen nieuwe bestanden verwerken, checkpoint per doeltabel |

**Switchen tussen full en incrementeel:** Eén `UPDATE` in de control table is genoeg —
geen codewijziging nodig.

```sql
UPDATE DEMO_DEV.CONFIG.pipeline_sources
SET    load_type = 'incremental'
WHERE  target_table = 'order_header'
```

**Checkpoints** (incremental mode): `{source_path}/_checkpoints/{target_table}/`

**Audit-kolommen per doeltabel:**

| Kolom | Bron |
|---|---|
| `_ingestion_timestamp` | `current_timestamp()` |
| `_source_system` | waarde uit control table |
| `_source_file` | `_metadata.file_path` |
| `_last_modified` | `_metadata.file_modification_time` |
| `_pipeline_run_id` | Workflow job run id (widget) |

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO_DEV", "Catalog")
dbutils.widgets.text("pipeline_run_id", "", "Pipeline Run ID (ingevuld door Workflow)")

catalog = dbutils.widgets.get("catalog")
pipeline_run_id = dbutils.widgets.get("pipeline_run_id")

print(f"Catalog          : {catalog}")
print(f"Pipeline run id  : {pipeline_run_id or '(niet opgegeven — handmatig run)'}")

## Stap 1 — Expliciete bron-schema's

Expliciete `StructType`-schema's per `target_table` voorkomen dat Spark Parquet
logical types moet inferen die het niet ondersteunt — met name `INT32 TIME(MILLIS,true)`
(de `[PARQUET_TYPE_ILLEGAL]` error op `SHIFT_START_TIME` / `SHIFT_END_TIME`).

**Bronze-principe:** deze laag schrijft de data zo dicht mogelijk bij de bron.
TIME-kolommen worden gelezen als `IntegerType` (milliseconden sinds middernacht)
en in die vorm opgeslagen — geen formattering of cast in de ingest-fase. Een column
`COMMENT` documenteert de eenheid voor downstream consumers; conversie naar
`HH:mm:ss` hoort thuis in de Silver/Integration-laag.

Doeltabellen die níet in `TARGET_SCHEMAS` staan vallen terug op schema-inferentie.
Gebruik `staging/schema_inspector.ipynb` om bron-schema's te diagnosticeren.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    DecimalType, DoubleType, IntegerType, StringType, TimestampType,
)

# Column COMMENT voor TIME-kolommen die als IntegerType doorvloeien.
# Delta neemt StructField metadata['comment'] over als column comment in de
# tabeldefinitie zodat downstream consumers de eenheid kunnen aflezen.
TIME_MILLIS_COMMENT = (
    "milliseconds since midnight (source: parquet INT32 TIME(MILLIS,true)); "
    "format to HH:mm:ss in the Silver/Integration layer"
)

# Expliciete bron-schema's per doeltabel.
# IntegerType voor SHIFT_START_TIME / SHIFT_END_TIME omdat parquet ze opslaat
# als INT32 TIME(MILLIS,true) — een logical type dat Spark weigert.
TARGET_SCHEMAS: dict[str, StructType] = {
    "order_header": StructType([
        StructField("ORDER_ID",                DecimalType(38, 0), True),
        StructField("TRUCK_ID",                DecimalType(38, 0), True),
        StructField("LOCATION_ID",             DoubleType(),       True),
        StructField("CUSTOMER_ID",             DecimalType(38, 0), True),
        StructField("DISCOUNT_ID",             StringType(),       True),
        StructField("SHIFT_ID",                DecimalType(38, 0), True),
        StructField("SHIFT_START_TIME",        IntegerType(),      True,
                    metadata={"comment": TIME_MILLIS_COMMENT}),
        StructField("SHIFT_END_TIME",          IntegerType(),      True,
                    metadata={"comment": TIME_MILLIS_COMMENT}),
        StructField("ORDER_CHANNEL",           StringType(),       True),
        StructField("ORDER_TS",                TimestampType(),    True),
        StructField("SERVED_TS",               StringType(),       True),
        StructField("ORDER_CURRENCY",          StringType(),       True),
        StructField("ORDER_AMOUNT",            DecimalType(38, 4), True),
        StructField("ORDER_TAX_AMOUNT",        StringType(),       True),
        StructField("ORDER_DISCOUNT_AMOUNT",   StringType(),       True),
        StructField("ORDER_TOTAL",             DecimalType(38, 4), True),
    ]),
    "order_detail": StructType([
        StructField("ORDER_DETAIL_ID",            DecimalType(38, 0), True),
        StructField("ORDER_ID",                   DecimalType(38, 0), True),
        StructField("MENU_ITEM_ID",               DecimalType(38, 0), True),
        StructField("DISCOUNT_ID",                StringType(),       True),
        StructField("LINE_NUMBER",                DecimalType(38, 0), True),
        StructField("QUANTITY",                   DecimalType(5, 0),  True),
        StructField("UNIT_PRICE",                 DecimalType(38, 4), True),
        StructField("PRICE",                      DecimalType(38, 4), True),
        StructField("ORDER_ITEM_DISCOUNT_AMOUNT", StringType(),       True),
    ]),
}

print(f"Bron-schema's gedefinieerd voor: {list(TARGET_SCHEMAS.keys())}")

## Stap 2 — Control table inlezen

In [ ]:
control_table = f"{catalog}.CONFIG.pipeline_sources"

sources_df = spark.sql(
    f"""
    SELECT *
    FROM   {control_table}
    WHERE  source_system = 'azurestorage'
    AND    is_active     = true
    """
)

source_rows = sources_df.collect()
print(f"Actieve azurestorage-bronnen gevonden: {len(source_rows)}")
for row in source_rows:
    print(f"  → {row['target_table']} ({row['load_type']}) — {row['file_pattern']}")

## Stap 3 — Ingest per bron

In [ ]:
from pyspark.sql import functions as F

row_count_report = []

for row in source_rows:
    source_path   = row["source_path"]
    file_pattern  = row["file_pattern"]
    target_schema = row["target_schema"]
    target_table  = row["target_table"]
    load_type     = row["load_type"]
    source_system = row["source_system"]

    full_target = f"{catalog}.{target_schema}.{target_table}"
    explicit_schema = TARGET_SCHEMAS.get(target_table)
    print(f"\n--- Verwerken: {full_target} (load_type={load_type}) ---")
    if explicit_schema is not None:
        print(f"  Bron-schema       : expliciet ({len(explicit_schema)} kolommen)")
    else:
        print("  Bron-schema       : geïnferred door Spark")

    if load_type == "full":
        # -----------------------------------------------------------------
        # FULL LOAD — lees alle bestanden die overeenkomen met het glob-filter
        # en overschrijf de doeltabel volledig.
        # -----------------------------------------------------------------
        reader = spark.read.format("parquet")
        if explicit_schema is not None:
            reader = reader.schema(explicit_schema)

        raw_df = (
            reader
            .option("pathGlobFilter", file_pattern)
            .option("recursiveFileLookup", "false")
            .load(source_path)
        )

        # Voeg de vijf audit-kolommen toe (CONTEXT.md §5).
        enriched_df = (
            raw_df
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_source_system",       F.lit(source_system))
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
            .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
            # _metadata is een struct-kolom die niet naar Delta mag worden geschreven.
            .drop("_metadata")
        )

        # Schrijf als Delta (full = overschrijven).
        (
            enriched_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(full_target)
        )

        written_count = spark.table(full_target).count()
        row_count_report.append((full_target, load_type, written_count))
        print(f"  [full] Geschreven: {written_count:,} rijen → {full_target}")

    elif load_type == "incremental":
        # -----------------------------------------------------------------
        # INCREMENTAL LOAD — Auto Loader (cloudFiles) met checkpoint.
        # Checkpoints leven onder _checkpoints/{target_table}/ binnen het
        # external volume.  Herdraaien pikt alleen nieuwe bestanden op;
        # er worden geen dubbele rijen geschreven.
        # -----------------------------------------------------------------
        checkpoint_path = f"{source_path}/_checkpoints/{target_table}"

        print(f"  Checkpoint pad   : {checkpoint_path}")
        print(f"  Bestandsfilter   : {file_pattern}")

        # Rijentelling vóór de run — voor delta row-count in de output.
        rows_before = (
            spark.table(full_target).count()
            if spark.catalog.tableExists(full_target)
            else 0
        )

        # Auto Loader leest incrementeel via cloudFiles.
        # trigger(availableNow=True) verwerkt alle beschikbare (nieuwe) bestanden
        # en stopt dan automatisch — geschikt voor batch Workflows.
        reader = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format",         "parquet")
            .option("cloudFiles.schemaLocation", checkpoint_path)
            .option("pathGlobFilter",            file_pattern)
            .option("recursiveFileLookup",       "false")
        )
        if explicit_schema is not None:
            reader = reader.schema(explicit_schema)

        stream_query = (
            reader
            .load(source_path)
            # Voeg audit-kolommen toe (CONTEXT.md §5).
            .withColumn("_ingestion_timestamp", F.current_timestamp())
            .withColumn("_source_system",       F.lit(source_system))
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
            .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
            .drop("_metadata")
            .writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", checkpoint_path)
            .option("mergeSchema",         "true")
            .trigger(availableNow=True)
            .toTable(full_target)
        )

        # Wacht tot alle beschikbare bestanden verwerkt zijn.
        stream_query.awaitTermination()

        rows_after  = spark.table(full_target).count()
        delta_count = rows_after - rows_before
        row_count_report.append((full_target, load_type, delta_count))
        print(f"  [incremental] Nieuwe rijen: {delta_count:,} | Totaal: {rows_after:,} → {full_target}")
        print(f"  Checkpoint opgeslagen onder: {checkpoint_path}")

    else:
        raise ValueError(
            f"Onbekend load_type='{load_type}' voor tabel '{target_table}'. "
            "Verwacht 'full' of 'incremental'."
        )

## Stap 4 — Validatie & row counts

In [ ]:
print("\n=== Row count samenvatting ===")
for table, mode, count in row_count_report:
    label = "geschreven" if mode == "full" else "nieuwe rijen"
    print(f"  [{mode}] {table}: {count:,} {label}")

## Resultaat

Alle actieve `azurestorage`-bronnen zijn ingeladen als Delta-tabellen in
`STAGING_AZURESTORAGE`. Elke tabel bevat de vijf audit-kolommen:
`_ingestion_timestamp`, `_source_system`, `_source_file`, `_last_modified`,
`_pipeline_run_id`.

**Full mode:** Alle bestanden opnieuw verwerkt, doeltabel volledig overschreven.

**Incremental mode:** Alleen nieuwe bestanden verwerkt via Auto Loader (`cloudFiles`).
Checkpoints staan onder `_checkpoints/{target_table}/` in het external volume.
Herdraaien levert uitsluitend de delta — geen dubbele rijen.

**Schema-aanpak (Bronze):** Doeltabellen in `TARGET_SCHEMAS` worden gelezen met een
expliciet `StructType`. TIME-kolommen (`SHIFT_START_TIME`, `SHIFT_END_TIME`) worden als
`IntegerType` ingelezen en als zodanig opgeslagen — milliseconden sinds middernacht.
Een column `COMMENT` op die kolommen documenteert de eenheid. De conversie naar
`HH:mm:ss` hoort thuis in de Silver/Integration-laag, niet in deze Bronze-ingest.

**Demo-tip:** Verander `load_type` in de control table met één `UPDATE` en draai het
Workflow opnieuw — de notebook past zijn gedrag automatisch aan zonder codewijziging.